# Feature Selection - Mutual Information & MRMR

This notebook performs feature selection on the insurance claims dataset using:
1. **Mutual Information** (sklearn) - measures dependency between features and target
2. **MRMR** (Minimum Redundancy Maximum Relevance) - selects features that are maximally relevant to the target while minimally redundant with each other

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
import mrmr
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Clean Data

In [ ]:
df = pd.read_csv('../data/raw/insurance_claims.csv')
print(f"Original shape: {df.shape}")
df.head()

In [ ]:
# Drop columns identified in data_overview as irrelevant/invalid
cols_to_drop = [
    'policy_number',       # identifier - no predictive value
    'policy_bind_date',    # date string - not useful directly
    'insured_zip',         # too many unique values, identifier-like
    'incident_location',   # too specific, high cardinality
    'incident_date',       # date string - not useful directly
    '_c39',                # 100% missing values (empty column)
]

df = df.drop(columns=cols_to_drop)
print(f"Shape after dropping irrelevant columns: {df.shape}")
print(f"\nDropped columns: {cols_to_drop}")

## 2. Handle Missing/Invalid Values

In [ ]:
# Replace '?' with NaN, then handle missing values
df.replace('?', np.nan, inplace=True)

# Check missing values after replacement
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Missing values after replacing '?':")
print(missing)
print()

# For categorical columns with missing values, fill with mode
for col in ['collision_type', 'property_damage', 'police_report_available', 'authorities_contacted']:
    if col in df.columns:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"{col}: filled {missing.get(col, 0)} missing values with mode '{mode_val}'")

print(f"\nRemaining missing values: {df.isnull().sum().sum()}")

## 3. Encode Categorical Features

In [ ]:
# Separate target
target_col = 'fraud_reported'
y = (df[target_col] == 'Y').astype(int)
X = df.drop(columns=[target_col])

# Label encode categorical features for MI calculation
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical features to encode: {len(categorical_cols)}")
print(f"Numerical features: {len(numerical_cols)}")
print(f"\nCategorical: {categorical_cols}")
print(f"\nNumerical: {numerical_cols}")

# Label encode
label_encoders = {}
X_encoded = X.copy()
for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
    label_encoders[col] = le

print(f"\nEncoded shape: {X_encoded.shape}")

## 4. Mutual Information Feature Selection

In [ ]:
# Identify which features are discrete (categorical) for MI calculation
discrete_mask = [col in categorical_cols for col in X_encoded.columns]

# Calculate mutual information scores
mi_scores = mutual_info_classif(
    X_encoded, y,
    discrete_features=discrete_mask,
    random_state=42,
    n_neighbors=5
)

# Create a DataFrame with MI scores
mi_df = pd.DataFrame({
    'Feature': X_encoded.columns,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False).reset_index(drop=True)

print("Mutual Information Scores (sorted):")
print("=" * 50)
for _, row in mi_df.iterrows():
    bar = '#' * int(row['MI_Score'] * 100)
    print(f"{row['Feature']:<35} {row['MI_Score']:.4f}  {bar}")

In [ ]:
# Visualize MI scores
plt.figure(figsize=(12, 8))
colors = ['#2ecc71' if score > 0.01 else '#e74c3c' for score in mi_df['MI_Score']]
plt.barh(range(len(mi_df)), mi_df['MI_Score'], color=colors)
plt.yticks(range(len(mi_df)), mi_df['Feature'])
plt.xlabel('Mutual Information Score')
plt.title('Feature Importance - Mutual Information')
plt.axvline(x=0.01, color='red', linestyle='--', label='Threshold (0.01)')
plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Features with very low MI (candidates for removal)
low_mi_features = mi_df[mi_df['MI_Score'] < 0.01]['Feature'].tolist()
print(f"\nFeatures with MI < 0.01 (low relevance): {low_mi_features}")

## 5. MRMR Feature Selection

MRMR selects features that have high relevance to the target but low redundancy among themselves.

In [ ]:
# Prepare DataFrame for MRMR (needs target as part of the dataframe)
df_mrmr = X_encoded.copy()
df_mrmr[target_col] = y

# Run MRMR - select top features
n_features_to_select = min(20, len(X_encoded.columns))

selected_features_mrmr = mrmr.mrmr_classif(
    X=df_mrmr.drop(columns=[target_col]),
    y=df_mrmr[target_col],
    K=n_features_to_select,
    show_progress=False
)

print(f"MRMR Selected Features (top {n_features_to_select}):")
print("=" * 50)
for i, feat in enumerate(selected_features_mrmr, 1):
    mi_score = mi_df[mi_df['Feature'] == feat]['MI_Score'].values[0]
    print(f"{i:>2}. {feat:<35} (MI: {mi_score:.4f})")

In [ ]:
# Compare MI ranking vs MRMR ranking
mi_top = mi_df.head(n_features_to_select)['Feature'].tolist()

print("Comparison: MI vs MRMR Rankings")
print("=" * 70)
print(f"{'Rank':<6} {'MI Top Features':<35} {'MRMR Top Features':<35}")
print("-" * 70)
for i in range(n_features_to_select):
    mi_feat = mi_top[i] if i < len(mi_top) else '-'
    mrmr_feat = selected_features_mrmr[i] if i < len(selected_features_mrmr) else '-'
    marker = ' *' if mi_feat != mrmr_feat else ''
    print(f"{i+1:<6} {mi_feat:<35} {mrmr_feat:<35}{marker}")

# Features in both
common = set(mi_top) & set(selected_features_mrmr)
only_mi = set(mi_top) - set(selected_features_mrmr)
only_mrmr = set(selected_features_mrmr) - set(mi_top)

print(f"\nFeatures in both: {len(common)}")
print(f"Only in MI top: {only_mi}")
print(f"Only in MRMR top: {only_mrmr}")

## 6. Final Feature Selection

In [ ]:
# Use MRMR selected features as the final selection
# (MRMR accounts for both relevance AND redundancy)
final_features = selected_features_mrmr

print(f"Final selected features ({len(final_features)}):")
print("=" * 50)
for i, feat in enumerate(final_features, 1):
    feat_type = 'categorical' if feat in categorical_cols else 'numerical'
    mi_score = mi_df[mi_df['Feature'] == feat]['MI_Score'].values[0]
    print(f"{i:>2}. {feat:<35} ({feat_type:<12}) MI={mi_score:.4f}")

# Save selected features to processed data
selected_df = df[final_features + [target_col]]
selected_df.to_csv('../data/processed/selected_features.csv', index=False)
print(f"\nSaved selected features dataset to data/processed/selected_features.csv")
print(f"Shape: {selected_df.shape}")

In [ ]:
# Visualize: side-by-side MI scores for selected vs dropped features
mi_df['Selected'] = mi_df['Feature'].isin(final_features)

plt.figure(figsize=(12, 8))
colors = ['#2ecc71' if sel else '#bdc3c7' for sel in mi_df['Selected']]
plt.barh(range(len(mi_df)), mi_df['MI_Score'], color=colors)
plt.yticks(range(len(mi_df)), mi_df['Feature'])
plt.xlabel('Mutual Information Score')
plt.title('Feature Selection Results (Green = Selected by MRMR)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Summary
dropped_features = [f for f in X_encoded.columns if f not in final_features]
print(f"\nDropped features ({len(dropped_features)}): {dropped_features}")